In [1]:
# 📘 i3d_tfhub_trainer.ipynb

# === Imports ===
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json

# === Load Preprocessed Data ===
X = np.load("X.npy") / 255.0  # Shape: (N, 100, 128, 128, 3)
y = np.load("y.npy")

with open("label_dict.json") as f:
    label_dict = json.load(f)
idx_to_label = {v: k for k, v in label_dict.items()}

# === Prepare Categorical Labels ===
from tensorflow.keras.utils import to_categorical
y_cat = to_categorical(y, num_classes=len(label_dict))

# === Resize Data to 224x224 ===
X_resized = tf.image.resize(X, [224, 224])  # Resize for I3D input

# === Split ===
X_train, X_test, y_train, y_test = train_test_split(X_resized, y_cat, test_size=0.2, random_state=42)

# === Load I3D model from TF Hub ===
i3d_url = "https://tfhub.dev/deepmind/i3d-kinetics-400/1"
i3d_layer = hub.KerasLayer(i3d_url, input_shape=(100, 224, 224, 3), trainable=False)

# === Model Definition ===
model = tf.keras.Sequential([
    i3d_layer,
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(label_dict), activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# === Train ===
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=2)

# === Save ===
model.save("i3d_tfhub_model.h5")

# === Evaluate ===
y_pred = model.predict(X_test).argmax(axis=1)
y_true = y_test.argmax(axis=1)

print(classification_report(y_true, y_pred, target_names=list(label_dict.keys())))

# === Confusion Matrix ===
plt.figure(figsize=(10,7))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True,
            xticklabels=label_dict.keys(), yticklabels=label_dict.keys(), fmt='d', cmap='Blues')
plt.title("📊 Confusion Matrix - I3D (TF Hub)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()


ModuleNotFoundError: No module named 'tensorflow_hub'